# DocLens 2/3: QLoRA fine-tune a vision-language model to output receipt JSON (Colab T4, ~75 min)

* Base: **Qwen2.5-VL-3B-Instruct**, 4-bit NF4 (bitsandbytes), via Unsloth, which makes VLM QLoRA fit a free T4
* LoRA r=16 on the language model's attention + MLP layers; the vision tower stays frozen
* Zero-shot on 50 test receipts → 1 epoch of fine-tuning on 800 train receipts → the same 50 test receipts

This one is OCR-free: it reads pixels and writes JSON directly. **Runtime → T4 GPU.**
Run notebook 1 first or in parallel (it writes `gt_test.jsonl`); this notebook writes its own copy as well.

In [ ]:
REPO_URL = "https://github.com/Harshchoubey55/doclens.git"
import os
os.chdir("/content")
if not os.path.exists("doclens"):
    !git clone -q {REPO_URL} doclens
os.chdir("/content/doclens")
!git pull -q
# unsloth pins compatible torch/transformers/trl/peft/bitsandbytes versions; install it first
!pip install -q unsloth
!pip install -q "pydantic>=2.5" "datasets>=2.19"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/doclens"
os.makedirs(f"{DRIVE}/predictions", exist_ok=True)
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    REPORT = "wandb"
except Exception:
    REPORT = "none"

EVAL_N = 50          # test receipts to generate on (≈20 s each on a T4); raise if you have time
MAX_SIDE = 768       # resize so a receipt costs ~500-750 visual tokens
MAX_NEW_TOKENS = 384

In [ ]:
from unsloth import FastVisionModel, is_bf16_supported  # import unsloth before transformers
import json, time, torch
from datasets import load_dataset
from doclens.cord import DATASET_ID
from doclens.extractors import resize_max
from doclens.metrics import evaluate
from doclens.predictions import write_jsonl
from doclens.schema import PROMPT, parse_model_output, restrict, to_target

ds = load_dataset(DATASET_ID)
gt_rows = [{"doc_id": f"test-{i}", "gt": restrict(json.loads(g)["gt_parse"])} for i, g in enumerate(ds["test"]["ground_truth"])]
write_jsonl("predictions/gt_test.jsonl", gt_rows)

model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit", load_in_4bit=True,
                                                   use_gradient_checkpointing="unsloth")
print(f"GPU memory after load: {torch.cuda.memory_allocated() / 2**30:.2f} GB")

In [ ]:
def generate(img):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(resize_max(img, MAX_SIDE), text, add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, use_cache=True, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def run_eval(name):
    FastVisionModel.for_inference(model)
    rows = []
    for i in range(EVAL_N):
        t0 = time.perf_counter()
        raw = generate(ds["test"][i]["image"])
        rows.append({"doc_id": f"test-{i}", "pred": parse_model_output(raw), "raw": raw,
                     "latency_s": round(time.perf_counter() - t0, 3)})
        if i % 10 == 0:
            print(i, raw[:160].replace("\n", " "))
    write_jsonl(f"predictions/{name}_test.jsonl", rows)
    s = evaluate([r["pred"] for r in rows], [g["gt"] for g in gt_rows[:EVAL_N]])
    print(name, {k: v for k, v in s.items() if k != "per_key_f1"})
    return s

zero_shot = run_eval("vlm_zeroshot")

## QLoRA fine-tuning

In [ ]:
model = FastVisionModel.get_peft_model(
    model, finetune_vision_layers=False, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)
model.print_trainable_parameters()

def to_conversation(ex):
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": resize_max(ex["image"], MAX_SIDE)},
                                     {"type": "text", "text": PROMPT}]},
        {"role": "assistant", "content": [{"type": "text", "text": to_target(json.loads(ex["ground_truth"])["gt_parse"])}]},
    ]}

train_conv = [to_conversation(ex) for ex in ds["train"]]
print(train_conv[0]["messages"][1]["content"][0]["text"][:300])

In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)
cfg = dict(per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=5, num_train_epochs=1,
           learning_rate=2e-4, fp16=not is_bf16_supported(), bf16=is_bf16_supported(), logging_steps=5,
           optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear", seed=3407, output_dir="runs/vlm",
           report_to=REPORT, run_name="qwen25vl3b-qlora-cord", remove_unused_columns=False, dataset_text_field="",
           dataset_kwargs={"skip_prepare_dataset": True})
# TRL renamed max_seq_length -> max_length; set whichever this version has
params = inspect.signature(SFTConfig.__init__).parameters
cfg["max_length" if "max_length" in params else "max_seq_length"] = 2048

trainer = SFTTrainer(model=model, tokenizer=tokenizer, data_collator=UnslothVisionDataCollator(model, tokenizer),
                     train_dataset=train_conv, args=SFTConfig(**cfg))
stats = trainer.train()
print(f"train time: {stats.metrics['train_runtime'] / 60:.1f} min, peak GPU {torch.cuda.max_memory_reserved() / 2**30:.1f} GB")

In [ ]:
finetuned = run_eval("vlm_qlora")
print(f"Field F1: zero-shot {zero_shot['field_f1']:.3f}  ->  QLoRA {finetuned['field_f1']:.3f}")
if REPORT == "wandb":
    import wandb
    wandb.log({"test/zeroshot_field_f1": zero_shot["field_f1"], "test/qlora_field_f1": finetuned["field_f1"]})
    wandb.finish()

In [ ]:
model.save_pretrained("doclens-qwen25vl-lora")      # adapter only (~100 MB)
tokenizer.save_pretrained("doclens-qwen25vl-lora")
!cp -r doclens-qwen25vl-lora {DRIVE}/
!cp predictions/*.jsonl {DRIVE}/predictions/
!ls {DRIVE}/predictions